# Atom Importance Distribution Analysis

This notebook answers: **for each combination of query type, model, and dataset, how
often is each atom (Atom 1, Atom 2, Atom 3) the most important one?**

**Our approach:**
1. Restrict to the first `K` iterations (`iteration < K`).
2. Group by `(dataset, model, query_type, query_id, target)` to collect those `K` rows belonging to the same query–answer pair.
3. **Average** each atom's Shapley value across the `K` iterations ($\bar\phi_a(e_i)$ in the paper).
4. Take the **argmax** of the averaged Shapley values to determine the most important atom $a^*_i$ for that query–answer pair.
5. Aggregate the resulting best-atom labels across all pairs to get the distribution.

**Output:**

- A summary DataFrame with counts and percentages per `(dataset, model, query_type)`.
- A single-column LaTeX table (percentages only) for the paper.

---
## 1. Imports and Configuration

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# Directory containing evaluation_*.csv files (this notebook lives in evaluations/)
EVALUATIONS_DIR = Path(".")

# Number of iterations to average over -- uses only the FIRST K iterations of
# each (type, query, target) group, matching report.py's aggregate(n_iterations=K)
# and the K=5 used throughout the rest of the paper (Section "Implementation Details").
K = 5

DATASET_NAME_MAP = {
    "FB":   "FB15k-237",
    "NELL": "NELL995",
}

# Number of atoms for each query type (used to know which shapley columns are valid)
ATOMS_PER_QUERY_TYPE = {
    "1p": 1,
    "2p": 2, "2i": 2, "2u": 2,
    "3p": 3, "3i": 3, "ip": 3, "pi": 3, "up": 3,
    "4p": 4, "4i": 4,
}

# Preferred display order for query types / models in the LaTeX table
QUERY_TYPE_ORDER = ["2p", "3p", "2i", "3i", "2u", "up", "ip", "pi"]
MODEL_ORDER       = ["cqd", "betae", "query2box", "gqe"]

# Display name overrides (internal code name -> paper display name)
QUERY_TYPE_DISPLAY = {
    "2p": "2p",  "3p": "3p",  "2i": "2i",  "3i": "3i",
    "2u": "2u",  "up": "2u1p", "ip": "1p2i", "pi": "2i1p",
}

MODEL_DISPLAY = {
    "betae":     "BetaE",
    "cqd":       "CQD",
    "gqe":       "GQE",
    "query2box": "Query2Box",
}

DATASET_DISPLAY = {
    "FB15k-237": "FB15k-237+H",
    "NELL995":   "NELL995+H",
}

## 2. Load All Evaluation CSV Files, Restrict to the First K Iterations

In [8]:
# Discover files
csv_files = sorted(EVALUATIONS_DIR.glob("evaluation_*.csv"))
print(f"Found {len(csv_files)} evaluation files")

all_dfs = []

for filepath in csv_files:
    # Filename format: evaluation_{DATASET}_{VERSION}_{MODEL}_{QUERY_TYPE}_{REL_SEL}.csv
    # Example:         evaluation_FB_2_betae_2u_random.csv
    parts = filepath.stem.split("_")  # ['evaluation', 'FB', '2', 'betae', '2u', 'random']

    dataset_code = parts[1]
    dataset_name = DATASET_NAME_MAP.get(dataset_code, dataset_code)

    df = pd.read_csv(filepath)

    # Restrict to the first K iterations, matching report.py's aggregate(n_iterations=K)
    df = df[df["iteration"] < K]

    df["dataset"] = dataset_name
    all_dfs.append(df)

# Concatenate all files; pandas fills missing columns (e.g. shapley2 absent in 2-atom CSVs) with NaN automatically.
raw = pd.concat(all_dfs, ignore_index=True)

print(f"\nTotal rows loaded (after K={K} cutoff): {len(raw):,}")
print(f"Columns: {list(raw.columns)}")

# ── Sanity checks ────────────────────────────────────────────────────────────
print("\nQuery types:", sorted(raw["type"].unique()))
print("Models:     ", sorted(raw["model"].unique()))
print("Datasets:   ", sorted(raw["dataset"].unique()))

group_sizes = raw.groupby(["dataset", "model", "type", "query", "target"]).size()
print(f"\nRows per (dataset, model, query_type, query_id, target) after the K={K} cutoff:")
print(group_sizes.value_counts().rename("# groups").rename_axis("K (iterations)"))

Found 64 evaluation files

Total rows loaded (after K=5 cutoff): 8,800,000
Columns: ['type', 'query', 'target', 'iteration', 'relation_selection', 'rel1', 'rel2', 'time', 'rank_empty', 'rank_full', 'shapley0', 'shapley1', 'rank_necc_atom0', 'rank_suff_atom0', 'rank_necc_atom1', 'rank_suff_atom1', 'best_atom_shapley', 'rank_necc_shapley', 'rank_suff_shapley', 'best_atom_random', 'rank_necc_random', 'rank_suff_random', 'best_atom_first', 'rank_necc_first', 'rank_suff_first', 'best_atom_last', 'rank_necc_last', 'rank_suff_last', 'model', 'dataset', 'rel3', 'shapley2', 'rank_necc_atom2', 'rank_suff_atom2']

Query types: ['2i', '2p', '2u', '3i', '3p', 'ip', 'pi', 'up']
Models:      ['betae', 'cqd', 'gqe', 'query2box']
Datasets:    ['FB15k-237', 'NELL995']

Rows per (dataset, model, query_type, query_id, target) after the K=5 cutoff:
K (iterations)
5    1760000
Name: # groups, dtype: int64


## 3. Average Shapley Values Across the K Iterations, Then Find the Best Atom

For every unique `(dataset, model, query_type, query_id, target)` group we compute the
**mean** of each atom's Shapley value across the K iterations ($\bar\phi_a(e_i)$), then
take the **argmax** over the atoms that actually exist for that query type ($a^*_i$).

In [9]:
# Identify Shapley columns
shapley_cols = sorted(
    [c for c in raw.columns if c.startswith("shapley") and c[7:].isdigit()],
    key=lambda c: int(c[7:])   # sort numerically: shapley0, shapley1, shapley2
)
print(f"Shapley columns detected: {shapley_cols}")

# Group and average
GROUP_KEYS = ["dataset", "model", "type", "query", "target"]

df_avg = (
    raw
    .groupby(GROUP_KEYS)[shapley_cols]
    .mean()               # K rows -> 1 row; NaN columns stay NaN (fewer-than-3-atom case)
    .reset_index()
)

print(f"\nRows after averaging (= unique query-answer pairs): {len(df_avg):,}")


# Best atom per query-answer pair
def find_best_atom(row: pd.Series, shapley_cols: list) -> float:
    """
    Return the 1-based index of the atom with the highest average Shapley value.

    Restricts comparison to the atoms that actually belong to this query type
    (looked up from ATOMS_PER_QUERY_TYPE), so NaN-padded columns from
    fewer-than-3-atom files are never mistakenly included.
    Returns NaN if all relevant Shapley values are NaN.
    """
    n_atoms = ATOMS_PER_QUERY_TYPE.get(row["type"], len(shapley_cols))
    values  = row[shapley_cols[:n_atoms]].values.astype(float)

    if np.all(np.isnan(values)):
        return np.nan

    # +1 converts from 0-based argmax to 1-based atom label (Atom 1, Atom 2, ...)
    return int(np.nanargmax(values)) + 1


df_avg["best_atom"] = df_avg.apply(find_best_atom, axis=1, shapley_cols=shapley_cols)

print("\nOverall best-atom distribution (1 = first atom, 2 = second, ...):")
print(df_avg["best_atom"].value_counts().sort_index())

Shapley columns detected: ['shapley0', 'shapley1', 'shapley2']

Rows after averaging (= unique query-answer pairs): 1,760,000

Overall best-atom distribution (1 = first atom, 2 = second, ...):
best_atom
1    385768
2    568162
3    806070
Name: count, dtype: int64


## 4. Compute Distribution per (Dataset, Model, Query Type)

In [10]:
dist_rows = []

for (dataset, model, query_type), group in df_avg.groupby(["dataset", "model", "type"]):
    n_total = len(group)
    n_atoms = ATOMS_PER_QUERY_TYPE.get(query_type, len(shapley_cols))

    atom_counts = group["best_atom"].value_counts().sort_index()

    row = {
        "dataset":    dataset,
        "model":      model,
        "query_type": query_type,
        "n_atoms":    n_atoms,
        "n_total":    n_total,
    }

    for atom_pos in range(1, n_atoms + 1):
        count = int(atom_counts.get(atom_pos, 0))
        row[f"atom{atom_pos}_count"] = count
        row[f"atom{atom_pos}_pct"]   = round(100.0 * count / n_total, 1) if n_total else 0.0

    dist_rows.append(row)

df_dist = pd.DataFrame(dist_rows)

qt_cat  = pd.CategoricalDtype(QUERY_TYPE_ORDER, ordered=True)
mdl_cat = pd.CategoricalDtype(MODEL_ORDER,       ordered=True)
df_dist["query_type"] = df_dist["query_type"].astype(qt_cat)
df_dist["model"]      = df_dist["model"].astype(mdl_cat)
df_dist = df_dist.sort_values(["n_atoms", "query_type", "model", "dataset"]).reset_index(drop=True)

# Row totals should match table-dataset.tex
print(df_dist.to_string(index=False))

  dataset     model query_type  n_atoms  n_total  atom1_count  atom1_pct  atom2_count  atom2_pct  atom3_count  atom3_pct
FB15k-237       cqd         2p        2    20000        10325       51.6         9675       48.4          NaN        NaN
  NELL995       cqd         2p        2    20000         8214       41.1        11786       58.9          NaN        NaN
FB15k-237     betae         2p        2    20000          317        1.6        19683       98.4          NaN        NaN
  NELL995     betae         2p        2    20000         3208       16.0        16792       84.0          NaN        NaN
FB15k-237 query2box         2p        2    20000         2162       10.8        17838       89.2          NaN        NaN
  NELL995 query2box         2p        2    20000         6286       31.4        13714       68.6          NaN        NaN
FB15k-237       gqe         2p        2    20000         1496        7.5        18504       92.5          NaN        NaN
  NELL995       gqe         2p  

## 5. Generate LaTeX Table (single-column, percentages only)

In [11]:
def build_latex_table_compact(
    df_dist: pd.DataFrame,
    datasets: list,
    models: list,
    query_types: list,
    caption: str,
    label: str,
    n_atoms_max: int = 3,
    dataset_display: dict = None,
    model_display: dict = None,
    qt_display: dict = None,
) -> str:
    """
    Single-column LaTeX table: percentages only, no raw counts.
    Uses tabular* + \\extracolsep{\\fill} (same pattern as tables/table-dataset.tex)
    so it fits \\columnwidth instead of spanning both columns.
    """
    dataset_display = dataset_display or {d: d for d in datasets}
    model_display   = model_display   or {m: m for m in models}
    qt_display      = qt_display      or {q: q for q in query_types}

    n_data_cols = n_atoms_max * len(datasets)
    col_spec = "@{}ll@{\\extracolsep{\\fill}}" + "r" * n_data_cols + "@{}"

    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(f"\\caption{{{caption}}}")
    lines.append(f"\\label{{{label}}}")
    lines.append(r"\footnotesize")
    lines.append(r"\setlength{\tabcolsep}{3pt}")
    lines.append(f"\\begin{{tabular*}}{{\\columnwidth}}{{{col_spec}}}")
    lines.append(r"\toprule")

    # ── Row 1: dataset multi-column headers ───────────────────────────────────
    header_parts   = ["", ""]
    cmidrule_parts = []
    col_idx = 3  # 1-based; cols 1-2 are Type and Model
    for i, ds in enumerate(datasets):
        ds_disp = dataset_display.get(ds, ds)
        header_parts.append(f"\\multicolumn{{{n_atoms_max}}}{{c}}{{\\textbf{{{ds_disp}}}}}")
        # Last block in the row: trim left only, so the rule reaches the true
        # right edge instead of stopping short (see page-reduction-suggestions.md).
        trim = "l" if i == len(datasets) - 1 else "lr"
        cmidrule_parts.append(f"\\cmidrule({trim}){{{col_idx}-{col_idx + n_atoms_max - 1}}}")
        col_idx += n_atoms_max
    lines.append(" & ".join(header_parts) + r" \\")
    lines.append(" ".join(cmidrule_parts))

    # ── Row 2: atom sub-headers ───────────────────────────────────────────────
    subheader_parts = [r"\textbf{Type}", r"\textbf{Model}"]
    for _ in datasets:
        for i in range(1, n_atoms_max + 1):
            subheader_parts.append(f"$a_{i}$")
    lines.append(" & ".join(subheader_parts) + r" \\")
    lines.append(r"\midrule")

    # ── Data rows ─────────────────────────────────────────────────────────────
    for qt in query_types:
        n_atoms_qt   = ATOMS_PER_QUERY_TYPE.get(qt, n_atoms_max)
        qt_disp      = qt_display.get(qt, qt)
        first_row_qt = True

        for model in models:
            mask = (df_dist["query_type"].astype(str) == qt) & \
                   (df_dist["model"].astype(str) == model)
            if not mask.any():
                continue

            row_parts = []
            if first_row_qt:
                row_parts.append(f"{{\\boldmath${qt_disp}$}}")
                first_row_qt = False
            else:
                row_parts.append("")
            row_parts.append(model_display.get(model, model))

            for ds in datasets:
                ds_mask = mask & (df_dist["dataset"] == ds)
                if not ds_mask.any():
                    row_parts.extend(["---"] * n_atoms_max)
                    continue

                r = df_dist[ds_mask].iloc[0]
                for atom_pos in range(1, n_atoms_max + 1):
                    if atom_pos > n_atoms_qt:
                        row_parts.append("---")
                    else:
                        pct = float(r.get(f"atom{atom_pos}_pct", 0.0))
                        row_parts.append(f"{pct:.1f}\\%")

            lines.append(" & ".join(row_parts) + r" \\")

        lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"
    lines.append(r"\end{tabular*}")
    lines.append(r"\end{table}")

    return "\n".join(lines)


# ── Build ──────────────────────────────────────────────────────────────────────
datasets_in_data = [d for d in ["FB15k-237", "NELL995"] if d in df_dist["dataset"].values]
models_in_data   = [m for m in MODEL_ORDER    if m in df_dist["model"].astype(str).values]
qt_in_data       = [qt for qt in QUERY_TYPE_ORDER if qt in df_dist["query_type"].astype(str).values]
n_atoms_max      = max(ATOMS_PER_QUERY_TYPE.get(qt, 1) for qt in qt_in_data)

latex_table = build_latex_table_compact(
    df_dist         = df_dist,
    datasets        = datasets_in_data,
    models          = models_in_data,
    query_types     = qt_in_data,
    n_atoms_max     = n_atoms_max,
    dataset_display = DATASET_DISPLAY,
    model_display   = MODEL_DISPLAY,
    qt_display      = QUERY_TYPE_DISPLAY,
    caption = (
        f"Atom importance distribution per query type, model, and dataset "
        f"($K={K}$). Each cell shows the percentage of query--answer pairs for which "
        f"the corresponding atom has the highest average Shapley value "
        r"$\bar\phi_a(e_i)$; percentages per row sum to $100\%$. Row totals equal the "
        r"number of query--answer pairs for that type and dataset, reported in "
        r"Table~\ref{tab:query-stats}. Cells marked with a dash (---) correspond to "
        "atom positions that do not exist for that query type."
    ),
    label = "tab:atom_dist",
)

print(latex_table)

\begin{table}[t]
\centering
\caption{Atom importance distribution per query type, model, and dataset ($K=5$). Each cell shows the percentage of query--answer pairs for which the corresponding atom has the highest average Shapley value $\bar\phi_a(e_i)$; percentages per row sum to $100\%$. Row totals equal the number of query--answer pairs for that type and dataset, reported in Table~\ref{tab:query-stats}. Cells marked with a dash (---) correspond to atom positions that do not exist for that query type.}
\label{tab:atom_dist}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular*}{\columnwidth}{@{}ll@{\extracolsep{\fill}}rrrrrr@{}}
\toprule
 &  & \multicolumn{3}{c}{\textbf{FB15k-237+H}} & \multicolumn{3}{c}{\textbf{NELL995+H}} \\
\cmidrule(lr){3-5} \cmidrule(l){6-8}
\textbf{Type} & \textbf{Model} & $a_1$ & $a_2$ & $a_3$ & $a_1$ & $a_2$ & $a_3$ \\
\midrule
{\boldmath$2p$} & CQD & 51.6\% & 48.4\% & --- & 41.1\% & 58.9\% & --- \\
 & BetaE & 1.6\% & 98.4\% & --- & 16.0\% & 84.0\% & --- \

## 6. Save

In [12]:
with open("table_atom_distribution.tex", "w") as f:
    f.write(latex_table + "\n")
print("Table saved to evaluations/table_atom_distribution.tex")

Table saved to evaluations/table_atom_distribution.tex
